In [111]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver_enriched")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_enriched.customer_dim (
        customer_sk BIGINT,
        customer_id STRING,
        first_name STRING,
        last_name STRING,
        city STRING,
        zip STRING,
        segment STRING,
        loyalty_tier STRING,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/enriched/customer_dim'
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS silver_enriched.batch_job_tracker (
        job_name STRING,
        last_processed_version BIGINT,
        run_timestamp TIMESTAMP,
        rows_processed BIGINT
    )
    USING DELTA
    LOCATION '/opt/spark-data/delta/silver/enriched/batch_job_tracker'
""")

spark.sql("""
    ALTER TABLE bronze.orders_raw
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("Tables ready ✅")

Tables ready ✅


In [112]:
JOB_NAME = "customer_dim_scd2"

last_version = get_last_processed_version(spark, JOB_NAME)
print(f"Last processed version: {last_version}")

NameError: name 'get_last_processed_version' is not defined

In [ ]:
bronze_df = read_bronze_cdf(spark, last_version)
new_row_count = bronze_df.count()
print(f"New rows to process: {new_row_count}")

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

if new_row_count > 0:
    customer_schema = StructType([
        StructField("order", StructType([
            StructField("order_time", StringType()),
            StructField("customer", StructType([
                StructField("customer_id", StringType()),
                StructField("name", StructType([
                    StructField("first_name", StringType()),
                    StructField("last_name", StringType())
                ])),
                StructField("address", StructType([
                    StructField("city", StringType()),
                    StructField("zip", StringType())
                ])),
                StructField("attributes", StructType([
                    StructField("segment", StringType()),
                    StructField("loyalty_tier", StringType())
                ]))
            ]))
        ]))
    ])

    parsed = bronze_df.withColumn("data", from_json(col("raw_payload"), customer_schema))

    incoming_customers = parsed.select(
        col("data.order.customer.customer_id").alias("customer_id"),
        col("data.order.customer.name.first_name").alias("first_name"),
        col("data.order.customer.name.last_name").alias("last_name"),
        col("data.order.customer.address.city").alias("city"),
        col("data.order.customer.address.zip").alias("zip"),
        col("data.order.customer.attributes.segment").alias("segment"),
        col("data.order.customer.attributes.loyalty_tier").alias("loyalty_tier"),
        col("data.order.order_time").cast("timestamp").alias("order_time")
    )

    w = Window.partitionBy("customer_id").orderBy(col("order_time").desc())

    latest_customers = (
        incoming_customers
        .withColumn("rn", row_number().over(w))
        .filter("rn = 1")
        .drop("rn", "order_time")
    )

    latest_customers.createOrReplaceTempView("incoming_customers")
    print(f"Unique customers to process: {latest_customers.count()}")
    latest_customers.show(truncate=False)
else:
    print("No new data. Skipping. ✅")

In [ ]:
if new_row_count > 0:
    existing_count = spark.sql("SELECT COUNT(*) FROM silver_enriched.customer_dim").collect()[0][0]

    if existing_count == 0:
        # ========== FIRST RUN: Seed ==========
        print("Seeding dimension...")
        spark.sql("""
            INSERT INTO silver_enriched.customer_dim
            SELECT
                monotonically_increasing_id() AS customer_sk,
                customer_id, first_name, last_name, city, zip, segment, loyalty_tier,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_customers
        """)
        print("Dimension seeded ✅")

    else:
        # ========== STEP 1: Close changed rows ==========
        print("Running SCD2 merge...")
        spark.sql("""
            MERGE INTO silver_enriched.customer_dim AS target
            USING incoming_customers AS source
            ON target.customer_id = source.customer_id AND target.is_current = true
            WHEN MATCHED AND (
                target.first_name   != source.first_name   OR
                target.last_name    != source.last_name    OR
                target.city         != source.city         OR
                target.zip          != source.zip          OR
                target.segment      != source.segment      OR
                target.loyalty_tier != source.loyalty_tier
            )
            THEN UPDATE SET
                target.is_current   = false,
                target.effective_to = current_timestamp()
        """)
        print("Step 1 done: Closed old versions.")

        # ========== STEP 2: Insert new versions ==========
        spark.sql("""
            INSERT INTO silver_enriched.customer_dim
            SELECT
                (SELECT COALESCE(MAX(customer_sk), 0) FROM silver_enriched.customer_dim)
                    + monotonically_increasing_id() + 1 AS customer_sk,
                src.customer_id, src.first_name, src.last_name,
                src.city, src.zip, src.segment, src.loyalty_tier,
                current_timestamp() AS effective_from,
                CAST('9999-12-31 00:00:00' AS TIMESTAMP) AS effective_to,
                true AS is_current
            FROM incoming_customers src
            LEFT ANTI JOIN silver_enriched.customer_dim dim
                ON src.customer_id = dim.customer_id AND dim.is_current = true
        """)
        print("Step 2 done: Inserted new versions ✅")

    # ========== SAVE VERSION BOOKMARK ==========
    current_version = get_current_bronze_version(spark)
    save_last_processed_version(spark, JOB_NAME, current_version, new_row_count)

In [ ]:
spark.sql("""
    SELECT customer_sk, customer_id, first_name, city, segment, loyalty_tier,
           is_current, effective_from, effective_to
    FROM silver_enriched.customer_dim
    ORDER BY customer_id, effective_from
""").toPandas()

In [ ]:
spark.sql("""
    SELECT * FROM silver_enriched.batch_job_tracker
    ORDER BY run_timestamp DESC
""").toPandas()